# Re-checking the myeloma mouse experiment (Rashid et al. 2018)

The paper's dose result: QPOP fitted a quadratic surface to 9 mouse groups and chose decitabine (Dec) 1.5 mg/kg + mitomycin C (MitoC) 1.5 mg/kg.
Every number below is copied from the paper's supplement: Table S15 (doses), S16 (outputs), S18 (published model).
Nothing is simulated. Notes and page references: `docs/mm2018_notes.md`.

In [1]:
import numpy as np
import plotly.graph_objects as go

# Table S15: dose per group (mg/kg)
MITO = np.array([0, 0, 2, 2, 0.5, 1.5, 1, 1, 1.0])
DEC = np.array([0, 1.5, 0, 1.5, 0.75, 0.75, 0.375, 1.125, 0.75])
# Table S16: mean survival (days), mean body-weight change (g), reported output
SURV = np.array([14, 18.5, 21, 30.1, 28, 23, 25.7, 28.2, 35.3])
WT = np.array([1.48, 0.44, 1.42, -3.67, -0.7, -1.9, 1.725, -1.55, 0.375])
OUT = np.array([15.5, 18.4, 22.4, 27.3, 27.3, 21.1, 26.7, 26.5, 38.4])
# Table S18: published coefficients for 1, MitoC, Dec, MitoC*Dec, MitoC^2, Dec^2
PAPER = np.array([14.9, 15.2, 10.0, 2.00, -6.41, -4.29])

def X(m, d):
    return np.stack([np.ones_like(m), m, d, m * d, m * m, d * d], axis=-1)

def fit(y, keep=slice(None)):
    b = np.linalg.lstsq(X(MITO[keep], DEC[keep]), y[keep], rcond=None)[0]
    r = y[keep] - X(MITO[keep], DEC[keep]) @ b
    return b, 1 - r @ r / ((y[keep] - y[keep].mean()) ** 2).sum()

GM, GD = np.meshgrid(np.linspace(0, 2, 81), np.linspace(0, 1.5, 61), indexing="ij")
def surface(b):
    return X(GM, GD) @ b
def optimum(b):
    z = surface(b); k = z.argmax()
    return GM.ravel()[k], GD.ravel()[k]

## 1. Does the output column add up?

Output is defined as survival + weight change (Table S16 header).

In [2]:
check = SURV + WT
for i in range(9):
    flag = "  <-- mismatch" if abs(check[i] - OUT[i]) > 0.1 else ""
    print(f"group {i+1}: {SURV[i]:5.1f} + {WT[i]:6.3f} = {check[i]:6.2f}   reported {OUT[i]:5.1f}{flag}")

group 1:  14.0 +  1.480 =  15.48   reported  15.5
group 2:  18.5 +  0.440 =  18.94   reported  18.4  <-- mismatch
group 3:  21.0 +  1.420 =  22.42   reported  22.4
group 4:  30.1 + -3.670 =  26.43   reported  27.3  <-- mismatch
group 5:  28.0 + -0.700 =  27.30   reported  27.3
group 6:  23.0 + -1.900 =  21.10   reported  21.1
group 7:  25.7 +  1.725 =  27.43   reported  26.7  <-- mismatch
group 8:  28.2 + -1.550 =  26.65   reported  26.5  <-- mismatch
group 9:  35.3 +  0.375 =  35.67   reported  38.4  <-- mismatch


## 2. Can the published model be rebuilt from the published data?

In [3]:
fits = {"paper (Table S18)": (PAPER, None), "refit: reported output": fit(OUT),
        "refit: survival + weight": fit(check), "refit: survival only": fit(SURV)}
for name, (b, r2) in fits.items():
    m, d = optimum(b)
    r2s = "0.680 (reported)" if r2 is None else f"{r2:.3f}"
    print(f"{name:26s} coef {np.round(b, 2)}  R2 {r2s:17s} best MitoC {m:.2f}, Dec {d:.2f}")
print("paper model at its own 9 groups:", np.round(X(MITO, DEC) @ PAPER, 1))
print("reported output:                ", OUT)

paper (Table S18)          coef [14.9  15.2  10.    2.   -6.41 -4.29]  R2 0.680 (reported)  best MitoC 1.43, Dec 1.50
refit: reported output     coef [16.54 20.47  0.22  0.67 -9.08  0.93]  R2 0.537             best MitoC 1.18, Dec 1.50
refit: survival + weight   coef [16.61 21.8  -2.46  0.18 -9.71  2.91]  R2 0.597             best MitoC 1.12, Dec 1.50
refit: survival only       coef [14.95 16.61  6.31  1.53 -7.09 -2.29]  R2 0.720             best MitoC 1.33, Dec 1.50
paper model at its own 9 groups: [14.9 20.2 19.7 31.  26.7 30.6 27.6 31.8 30.3]
reported output:                 [15.5 18.4 22.4 27.3 27.3 21.1 26.7 26.5 38.4]


In [4]:
fig = go.Figure()
for i, (name, (b, _)) in enumerate(fits.items()):
    fig.add_trace(go.Surface(x=np.linspace(0, 2, 81), y=np.linspace(0, 1.5, 61), z=surface(b).T,
                             colorscale="Viridis", showscale=False, opacity=0.8, visible=i == 0, name=name))
fig.add_trace(go.Scatter3d(x=MITO, y=DEC, z=OUT, mode="markers", marker=dict(size=5, color="black"),
                           name="reported group outputs"))
n = len(fits)
fig.update_layout(updatemenus=[dict(buttons=[dict(label=name, method="update",
                  args=[dict(visible=[j == i for j in range(n)] + [True])]) for i, name in enumerate(fits)])],
                  scene=dict(xaxis_title="MitoC (mg/kg)", yaxis_title="Dec (mg/kg)", zaxis_title="output (days)"),
                  title="Published surface vs refits of the published data (pick one, rotate)", height=620)
fig

## 3. Leave-one-out: drop one mouse group, refit, predict the dropped group

With 6 coefficients and 9 groups, the fit has only 3 points to spare. If the surface is real, dropping one group should barely move it.

In [5]:
loo = []
for i in range(9):
    keep = np.arange(9) != i
    b, _ = fit(OUT, keep)
    loo.append((i + 1, *optimum(b), float(X(MITO[i], DEC[i]) @ b), OUT[i], b))
for g, m, d, pred, obs, _ in loo:
    print(f"drop group {g}: best MitoC {m:.2f}, Dec {d:.2f}   predicted {pred:6.1f}  observed {obs:5.1f}")
err = np.array([p - o for _, _, _, p, o, _ in loo])
print(f"leave-one-out RMSE: {np.sqrt((err**2).mean()):.1f} days; SD of outputs: {OUT.std(ddof=1):.1f} days")

drop group 1: best MitoC 0.53, Dec 0.00   predicted   33.8  observed  15.5
drop group 2: best MitoC 0.93, Dec 1.50   predicted   28.3  observed  18.4
drop group 3: best MitoC 1.18, Dec 1.50   predicted    0.8  observed  22.4
drop group 4: best MitoC 1.05, Dec 0.20   predicted   -2.7  observed  27.3
drop group 5: best MitoC 1.15, Dec 1.50   predicted   23.6  observed  27.3
drop group 6: best MitoC 2.00, Dec 0.80   predicted   35.3  observed  21.1
drop group 7: best MitoC 1.12, Dec 1.50   predicted   30.1  observed  26.7
drop group 8: best MitoC 1.10, Dec 1.50   predicted   33.6  observed  26.5
drop group 9: best MitoC 1.23, Dec 1.50   predicted   26.0  observed  38.4
leave-one-out RMSE: 15.7 days; SD of outputs: 6.6 days


In [6]:
fig = go.Figure()
for g, m, d, pred, obs, b in loo:
    fig.add_trace(go.Surface(x=np.linspace(0, 2, 81), y=np.linspace(0, 1.5, 61), z=surface(b).T,
                             colorscale="Viridis", showscale=False, opacity=0.8, visible=g == 1))
fig.add_trace(go.Scatter3d(x=MITO, y=DEC, z=OUT, mode="markers", marker=dict(size=5, color="black"), name="groups"))
fig.add_trace(go.Scatter3d(x=[1.5], y=[1.5], z=[33], mode="markers", marker=dict(size=7, symbol="diamond", color="red"),
                           name="paper's chosen dose"))
steps = [dict(method="update", label=str(g), args=[dict(visible=[j == g - 1 for j in range(9)] + [True, True])])
         for g, *_ in loo]
fig.update_layout(sliders=[dict(active=0, currentvalue=dict(prefix="dropped group: "), steps=steps)],
                  scene=dict(xaxis_title="MitoC (mg/kg)", yaxis_title="Dec (mg/kg)", zaxis_title="output (days)",
                             zaxis=dict(range=[-10, 50])),
                  title="Refit without one group (use the slider)", height=620)
fig

In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=[m for _, m, *_ in loo], y=[d for _, _, d, *_ in loo], mode="markers+text",
                         text=[f"drop {g}" for g, *_ in loo], textposition="top center", name="leave-one-out optimum"))
fig.add_trace(go.Scatter(x=[1.5], y=[1.5], mode="markers", marker=dict(size=12, symbol="diamond"), name="paper"))
fig.add_trace(go.Scatter(x=MITO, y=DEC, mode="markers", marker=dict(symbol="x", size=9), name="tested doses"))
fig.update_layout(xaxis_title="MitoC (mg/kg)", yaxis_title="Dec (mg/kg)", height=450,
                  title="Where the 'best dose' lands when one group is left out")
fig

## What this says

1. The output column is wrong in 5 of 9 rows (4 by more than 0.5 days). The largest error is the centre group (35.67 computed vs 38.4 reported), which is the best group and pulls the surface's peak.
2. The published coefficients cannot be rebuilt from the published outputs. The refit has R² 0.54, not 0.68, and a different shape.
3. Leave-one-out moves the "best dose" across most of the tested range. Prediction error for the dropped group (RMSE 15.7 days) is more than twice the spread of the outputs (SD 6.6 days): the model predicts new groups worse than simply guessing the average.
4. Every fit puts Dec at the highest tested dose (1.5 mg/kg), so the optimum, if there is one, may lie outside the tested range.
5. Open question for the authors: the raw survival data (Table S19, Fig 5B) record fewer than 4 deaths per group. How censored mice were handled is not stated.

Conclusion: the paper's in vivo dose "optimization" is not supported by its own data.